# AIMLC ZG5536 — Large Language Models for Generative AI
#### Assignment 2B — RAG Pipeline (Group 157)
**Domain:** Indian Financial Markets and Digital Payments (RBI / SEBI documents)

This notebook builds a small, complete RAG system on the Assignment 1A corpus: choose a chunk size, compare dense / BM25 / hybrid search, rerank, then generate grounded answers and check them with an LLM judge.

**Working method:** one stage at a time. Each stage is explained, run by the operator, and verified before the next one is added.

## Requirements map (20 marks)

| Part | What is built | Marks | Notebook stage |
|---|---|---|---|
| A | Dense search (FAISS) and chunk-size choice (100 vs 400 tokens) | 5 | Stages 1–3 |
| B | Dense vs BM25 vs hybrid (RRF) | 4 | Stage 4 |
| C | Cross-encoder reranking | 4 | Stage 5 |
| D | Grounded generation + LLM judge | 7 | Stages 6–7 |

**Reused from Assignment 1A (no new data):** the cleaned domain `.txt` files, 10 queries with expected answers from the 1A evaluation split, and the 1A fine-tuned model as the answer generator.

**Requirements authority:** `assignment-2b/concepts/Assignment 2B - RAG Pipeline.pdf`.
**Known PDF inconsistency:** the text says chunk sizes 100 and 400, one table says 100 and 500. This notebook uses **100 and 400** (the text) and discloses the discrepancy in Stage 3.

## How 1A connects to 2B

| Step | What changes | What it does **not** do |
|---|---|---|
| **CPT** (continued pretraining) | All weights keep learning next-token prediction on the finance corpus, so the model's word predictions fit the domain | It does not teach question/answer behaviour |
| **Instruction SFT** | Trains on question→answer pairs so the model answers briefly in the expected style | It does not give the model access to documents at answer time |
| **LoRA / QLoRA** | Trains small low-rank adapter matrices while the base weights stay frozen; QLoRA also stores the frozen base in 4-bit | It is a cheaper way to do SFT, not a different goal |
| **RAG** (this assignment) | At question time, retrieve relevant passages and put them in the prompt so the answer can be grounded in text | It retrains nothing |

A lower perplexity in 1A shows the model predicts domain text better. It does **not** show that the model answers questions correctly or stays faithful to supplied passages. Part D tests that directly.

---
## Stage 0 — Artifact check, environment and configuration

**Goal:** confirm that everything 2B reuses from 1A is present and unchanged, record the compute resources, and fix one configuration that every later stage uses. No retrieval or model loading happens here.

| Cell | Purpose |
|---|---|
| 0A | Configuration: paths, seed, constants; locate this repository's `data/` folder |
| 0B | Resources and package versions (GPU, VRAM, host RAM, disk) |
| 0C | Data integrity: repository `data/` against its SHA-256 manifest, and against the 1A copies on the volume |
| 0D | Model artifacts (CPT model, adapter) checked against the committed hash manifest |
| 0E | Load the corpus and save an input manifest |
| 0F | Model access and disk budget (no downloads yet) |
| 0G | Gate 0 summary |

In [9]:
# Stage 0A: Configuration and path discovery
import os, sys, json, random, hashlib, time, platform
from pathlib import Path

def stage_header(title, width=None):
    """Every cell's printed output starts with a stage header and an underline."""
    print(title)
    print("-" * (width or max(len(title), 10)))

SEED = 42                                   # same seed family as the 1A split (Stage 8E)
random.seed(SEED)

# This repository holds a frozen copy of the 1A text data under data/ (verified by data/MANIFEST.sha256).
# The 1A volume (mounted at /home/group157) still holds the model weights, which are too large for git.
def find_repo_root():
    # Look for data/MANIFEST.sha256 starting from the working directory, then common clone locations.
    # os.getcwd() raises if the kernel's start folder was deleted or re-cloned; restart the kernel in that case.
    starts = []
    try:
        cwd = Path(os.getcwd())
        starts += [cwd, *cwd.parents]
    except FileNotFoundError:
        print("WARNING: the kernel's working directory no longer exists (folder deleted or re-cloned).")
        print("         Restart the kernel from the repository folder. Searching common locations instead.")
    if os.environ.get("REPO_ROOT"):
        starts.append(Path(os.environ["REPO_ROOT"]))
    starts += [Path.home() / "llm-rag", Path("/home/group157/llm-rag")]
    for base in starts:
        if (base / "data" / "MANIFEST.sha256").exists():
            return base
    return None

REPO_ROOT = find_repo_root()
assert REPO_ROOT is not None, "data/MANIFEST.sha256 not found: run this notebook from inside the cloned repository"
DATA_DIR      = REPO_ROOT / "data"
CORPUS_DIR    = DATA_DIR / "corpus"
QA_FILE       = DATA_DIR / "qa" / "instruction_dataset.jsonl"            # as submitted in 1A (Q/A only)
CURATED_FILE  = DATA_DIR / "qa" / "instruction_dataset_curated.jsonl"    # adds source_document and evidence

ARTIFACT_ROOT = Path(os.environ.get("ARTIFACT_ROOT", "/home/group157"))  # 1A volume: weights + original copies
PV_CORPUS_DIR = ARTIFACT_ROOT / "domain_corpus"                          # read-only cross-check, never written
PV_QA_FILE    = ARTIFACT_ROOT / "instruction_dataset.jsonl"              # read-only cross-check, never written
TRAIN_FILE    = DATA_DIR / "qa" / "instruction_dataset_train.jsonl"      # 1A split export (957)
EVAL_FILE     = DATA_DIR / "qa" / "instruction_dataset_eval.jsonl"       # 1A split export (239)
MODELS_DIR    = REPO_ROOT / "models"
MODELS_MANIFEST = MODELS_DIR / "MANIFEST.sha256"                         # hashes of CPT weights and adapter
ADAPTER_DIR   = MODELS_DIR / "qlora_finance_adapter"                     # committed in this repository (about 11 MB)
PV_ADAPTER_DIR = ARTIFACT_ROOT / "qlora_finance_adapter"                 # 1A volume copy, cross-check only
# CPT weights (1.45 GB, not in git). Order: CPT_DIR environment variable, then the read-only backup, then the original.
_cpt_candidates = [os.environ.get("CPT_DIR"),
                   ARTIFACT_ROOT / "model-backups" / "cpt_model_1a",
                   ARTIFACT_ROOT / "cpt_model"]
CPT_DIR = next((Path(c) for c in _cpt_candidates if c and (Path(c) / "model.safetensors").exists()),
               ARTIFACT_ROOT / "cpt_model")
WORK_DIR      = ARTIFACT_ROOT / "assignment_2b_outputs"                  # 2B generated artifacts (not in git)
WORK_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_DOCS = 30
EXPECTED_QA   = {"total": 1196, "train": 957, "eval": 239}  # from saved 1A outputs (Stage 8E)

# Retrieval constants, fixed by the assignment PDF
CHUNK_SIZES   = [100, 400]       # tokens, no overlap (text of PDF; its table says 500, see intro)
TOP_K_CHUNK_EVAL = 3
TOP_K_COMPARE = 5
TOP_K_FUSE    = 20
RRF_K         = 60
TOP_K_RERANK  = 10
TOP_K_FINAL   = 3

print("Stage 0A: configuration")
print("-" * 40)
print(f"Python         : {platform.python_version()}")
try:
    print(f"Working dir    : {Path(os.getcwd()).resolve()}")
except FileNotFoundError:
    print("Working dir    : (deleted: restart the kernel from the repository folder)")
print(f"Repository root: {REPO_ROOT.resolve()}")
print(f"Data dir       : {DATA_DIR.resolve()}")
print(f"ARTIFACT_ROOT  : {ARTIFACT_ROOT.resolve() if ARTIFACT_ROOT.exists() else ARTIFACT_ROOT}  (exists={ARTIFACT_ROOT.exists()})")
print(f"2B output dir  : {WORK_DIR.resolve()}")

print("\nTop level of ARTIFACT_ROOT (1A volume):")
if ARTIFACT_ROOT.exists():
    for p in sorted(ARTIFACT_ROOT.iterdir()):
        kind = "dir " if p.is_dir() else "file"
        print(f"  {kind}  {p.name}")
else:
    print("  (missing)")

STAGE0 = {}   # check name -> (status, detail), filled by later Stage 0 cells
print("\n>> [OK] Configuration set.")

Stage 0A: configuration
----------------------------------------
Python         : 3.11.11
Working dir    : /home/group157/llm-assignment-2b/llm-rag
Repository root: /home/group157/llm-assignment-2b/llm-rag
Data dir       : /home/group157/llm-assignment-2b/llm-rag/data
ARTIFACT_ROOT  : /home/group157  (exists=True)
2B output dir  : /home/group157/assignment_2b_outputs

Top level of ARTIFACT_ROOT (1A volume):
  dir   .Trash-1000
  dir   .ipynb_checkpoints
  dir   assignment_2b_outputs
  dir   conv-ai
  file  cpt_loss.png
  dir   cpt_model
  dir   cpt_output
  dir   domain_corpus
  file  instruction_dataset.jsonl
  file  instruction_dataset_eval.jsonl
  file  instruction_dataset_train.jsonl
  dir   llm-assignment-1a-finance-corpus
  dir   llm-assignment-2b
  file  llm_baseline_requirements.txt
  dir   model-backups
  dir   qlora_finance_adapter
  dir   qlora_training_output
  file  rag_requirements.txt
  dir   venv
  dir   venv-rag

>> [OK] Configuration set.


In [20]:
# Stage 0B: Resources and package versions
import shutil, importlib, subprocess

print("Stage 0B: compute resources")
print("-" * 40)

# --- GPU ---
gpu_info = {"available": False}
try:
    import torch
    gpu_info["torch"] = torch.__version__
    gpu_info["cuda_build"] = torch.version.cuda
    gpu_info["available"] = torch.cuda.is_available()
    if gpu_info["available"]:
        props = torch.cuda.get_device_properties(0)
        free_b, total_b = torch.cuda.mem_get_info(0)
        gpu_info.update(name=props.name, total_gb=round(total_b/1e9, 1), free_gb=round(free_b/1e9, 1),
                        bf16=torch.cuda.is_bf16_supported())
except Exception as e:
    gpu_info["error"] = repr(e)
print("GPU            :", gpu_info)

# --- Host RAM (physical and the container limit, which is what actually binds) ---
def read_first(paths):
    for p in paths:
        try:
            return Path(p).read_text().strip()
        except Exception:
            pass
    return None

mem_total_kb = None
try:
    for line in Path("/proc/meminfo").read_text().splitlines():
        if line.startswith("MemTotal"):
            mem_total_kb = int(line.split()[1])
        if line.startswith("MemAvailable"):
            mem_avail_kb = int(line.split()[1])
except Exception:
    mem_avail_kb = None
cg_limit = read_first(["/sys/fs/cgroup/memory.max", "/sys/fs/cgroup/memory/memory.limit_in_bytes"])
cg_limit_gb = None
if cg_limit and cg_limit.isdigit() and int(cg_limit) < 1 << 60:
    cg_limit_gb = round(int(cg_limit)/1e9, 1)
print(f"Host RAM       : machine total {round(mem_total_kb/1e6,1) if mem_total_kb else '?'} GB | "
      f"available {round(mem_avail_kb/1e6,1) if mem_avail_kb else '?'} GB | container limit {cg_limit_gb or 'none/unknown'} GB")

# --- Persistent disk ---
disk_target = ARTIFACT_ROOT if ARTIFACT_ROOT.exists() else REPO_ROOT
du = shutil.disk_usage(disk_target)
print(f"Persistent disk: {disk_target}  total {du.total/1e9:.1f} GB | free {du.free/1e9:.1f} GB")
hf_home = Path(os.environ.get("HF_HOME", Path.home() / ".cache" / "huggingface"))
print(f"HF cache dir   : {hf_home}  (exists={hf_home.exists()})")

# --- Package versions ---
PKGS = ["numpy", "torch", "transformers", "tokenizers", "peft", "bitsandbytes", "accelerate",
        "sentence_transformers", "faiss", "rank_bm25", "huggingface_hub", "pandas"]
versions = {}
for name in PKGS:
    try:
        m = importlib.import_module(name)
        versions[name] = getattr(m, "__version__", "installed (no __version__)")
    except Exception:
        versions[name] = None
print("\nPackage versions:")
for k, v in versions.items():
    print(f"  {k:<22}{v if v else 'MISSING'}")

missing = [k for k, v in versions.items() if v is None]
print("\nMissing packages:", missing or "none")
print("If packages are missing: run kubeflow/setup_env.sh in a terminal and select the \"Python (RAG venv)\" kernel.")

(WORK_DIR / "environment_report.json").write_text(json.dumps(
    {"gpu": gpu_info, "mem_total_kb": mem_total_kb, "container_limit_gb": cg_limit_gb,
     "disk_free_gb": round(du.free/1e9, 1), "versions": versions, "python": platform.python_version()}, indent=2))
STAGE0["resources"] = ("PASS" if gpu_info.get("available") else "WARN", f"gpu={gpu_info.get('name','none')}")
print("\n>> [OK] Saved assignment_2b_outputs/environment_report.json")

Stage 0B: compute resources
----------------------------------------
GPU            : {'available': True, 'torch': '2.5.1+cu124', 'cuda_build': '12.4', 'name': 'NVIDIA L40S', 'total_gb': 47.8, 'free_gb': 39.4, 'bf16': True}
Host RAM       : machine total 528.0 GB | available 500.3 GB | container limit 19.3 GB
Persistent disk: /home/group157  total 16002.9 GB | free 5883.0 GB
HF cache dir   : /home/jovyan/.cache/huggingface  (exists=True)

Package versions:
  numpy                 2.4.6
  torch                 2.5.1+cu124
  transformers          4.46.3
  tokenizers            0.20.3
  peft                  0.13.2
  bitsandbytes          0.44.1
  accelerate            1.1.1
  sentence_transformers 3.3.1
  faiss                 1.9.0
  rank_bm25             installed (no __version__)
  huggingface_hub       0.26.2
  pandas                2.2.3

Missing packages: none
If packages are missing: run kubeflow/setup_env.sh in a terminal and select the "Python (RAG venv)" kernel.

>> [OK] Saved 

In [ ]:
# Stage 0B-install: optional, run only when a package is missing or a new one is needed.
stage_header("Stage 0B-install: optional package install")
# Normal route: add the package to kubeflow/requirements.txt and re-run `bash kubeflow/setup_env.sh`
# in a terminal. This cell is the in-notebook shortcut and installs into the SAME venv (the kernel's
# interpreter), so the package persists on the volume.
INSTALL_MISSING = False        # set True, run this cell once, then restart the kernel and re-run from 0A
EXTRA_PACKAGES = []            # e.g. ["rouge-score==0.1.2"]; also add each one to kubeflow/requirements.txt

REQ_PATH = REPO_ROOT / "kubeflow" / "requirements.txt"
if INSTALL_MISSING:
    cmd = [sys.executable, "-m", "pip", "install", "-r", str(REQ_PATH), *EXTRA_PACKAGES]
    print("Running:", " ".join(cmd), "\n(packages that are already satisfied are skipped)\n")
    r = subprocess.run(cmd, capture_output=True, text=True)
    print((r.stdout if r.returncode == 0 else r.stderr)[-1500:])
    print("Done." if r.returncode == 0 else "pip FAILED, see the message above.")
    if EXTRA_PACKAGES:
        print("\nREMINDER: add these to kubeflow/requirements.txt so setup_env.sh and check_env.py include them:", EXTRA_PACKAGES)
    print("Restart the kernel, then re-run from Stage 0A so the new packages are imported.")
else:
    print("INSTALL_MISSING is False: nothing installed.")

In [22]:
# Stage 0C: Data integrity (repository data/ vs its manifest, and vs the 1A copies on the volume)
def sha256_file(p, chunk=1 << 20):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

def human(n):
    for unit in ["B", "KB", "MB", "GB"]:
        if n < 1024 or unit == "GB":
            return f"{n:.1f} {unit}"
        n /= 1024

def jsonl_rows(p):
    with open(p, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

print("Stage 0C: data integrity")
print("-" * 40)

# 1) Repository data against data/MANIFEST.sha256 (detects any edit to the committed copy)
expected = {}
for line in (DATA_DIR / "MANIFEST.sha256").read_text(encoding="utf-8").splitlines():
    if line.strip():
        digest, rel = line.split("  ", 1)
        expected[rel] = digest
bad, missing_files = [], []
for rel, digest in expected.items():
    p = DATA_DIR / rel
    if not p.exists():
        missing_files.append(rel)
    elif sha256_file(p) != digest:
        bad.append(rel)
on_disk = {p.relative_to(DATA_DIR).as_posix() for p in DATA_DIR.rglob("*") if p.is_file()} - {"MANIFEST.sha256", "README.md"}
extra = sorted(on_disk - set(expected))
print(f"Manifest entries: {len(expected)} | missing {len(missing_files)} | changed {len(bad)} | extra {len(extra)}")
for tag, items in [("missing", missing_files), ("CHANGED", bad), ("extra", extra)]:
    for r in items[:10]:
        print(f"  {tag}: {r}")
STAGE0["repo_data_manifest"] = ("PASS" if not (bad or missing_files or extra) else "FAIL",
                                f"{len(expected)} files checked")

corpus_files = sorted(CORPUS_DIR.glob("*.txt"))
STAGE0["corpus"] = ("PASS" if len(corpus_files) == EXPECTED_DOCS else "FAIL", f"{len(corpus_files)} files")

# 2) Q/A files: counts, schema, ordered identity between the two files
qa_rows, cur_rows = jsonl_rows(QA_FILE), jsonl_rows(CURATED_FILE)
same_order = (len(qa_rows) == len(cur_rows) and
              all(a["instruction"] == b["instruction"] and a["response"] == b["response"]
                  for a, b in zip(cur_rows, qa_rows)))
print(f"\nQ/A rows: submitted {len(qa_rows)} | curated {len(cur_rows)} (expected {EXPECTED_QA['total']})")
print("Submitted keys:", sorted(qa_rows[0]), "| curated keys:", sorted(cur_rows[0]))
print("Ordered Q/A identical across the two files:", same_order)
STAGE0["qa_rows"] = ("PASS" if len(qa_rows) == EXPECTED_QA["total"] == len(cur_rows) and same_order else "FAIL",
                     f"{len(qa_rows)} rows")

# 3) Cross-check the 1A volume copies (read-only). A difference means the volume was overwritten or differs.
pv_status = []
if PV_CORPUS_DIR.exists():
    same = diff = absent = 0
    for p in corpus_files:
        q = PV_CORPUS_DIR / p.name
        if not q.exists():
            absent += 1
        elif sha256_file(q) == sha256_file(p):
            same += 1
        else:
            diff += 1
            print("  DIFFERS from volume:", p.name)
    pv_extra = sorted(set(x.name for x in PV_CORPUS_DIR.glob("*.txt")) - set(p.name for p in corpus_files))
    print(f"\nVolume corpus ({PV_CORPUS_DIR}): identical {same} | differs {diff} | absent on volume {absent} | extra on volume {len(pv_extra)}")
    pv_status.append("corpus")
    STAGE0["volume_corpus_matches_repo"] = ("PASS" if diff == 0 and absent == 0 and not pv_extra else "WARN", f"same={same} diff={diff}")
else:
    print(f"\nVolume corpus not present at {PV_CORPUS_DIR} (fine: the repository copy is authoritative).")
    STAGE0["volume_corpus_matches_repo"] = ("PENDING", "no volume copy")
if PV_QA_FILE.exists():
    ok = sha256_file(PV_QA_FILE) == sha256_file(QA_FILE)
    print(f"Volume Q/A file identical to repository copy: {ok}")
    STAGE0["volume_qa_matches_repo"] = ("PASS" if ok else "WARN", "")

# 4) Split exports: counts, disjoint, union equals the full file
train_rows, eval_rows = jsonl_rows(TRAIN_FILE), jsonl_rows(EVAL_FILE)
qkey = lambda r: (r["instruction"], r["response"])
tr_set, ev_set, full_set = set(map(qkey, train_rows)), set(map(qkey, eval_rows)), set(map(qkey, qa_rows))
split_ok = (len(train_rows) == EXPECTED_QA["train"] and len(eval_rows) == EXPECTED_QA["eval"]
            and not (tr_set & ev_set) and (tr_set | ev_set) == full_set)
print(f"\nSplit exports: train {len(train_rows)} (expected {EXPECTED_QA['train']}) | eval {len(eval_rows)} (expected {EXPECTED_QA['eval']})"
      f" | overlap {len(tr_set & ev_set)} | union equals full file: {(tr_set | ev_set) == full_set}")
STAGE0["split_exports"] = ("PASS" if split_ok else "FAIL", f"{len(train_rows)}/{len(eval_rows)}")
print("\n>> [OK] Data integrity checked.")

Stage 0C: data integrity
----------------------------------------
Manifest entries: 34 | missing 0 | changed 0 | extra 0

Q/A rows: submitted 1196 | curated 1196 (expected 1196)
Submitted keys: ['instruction', 'response'] | curated keys: ['evidence', 'instruction', 'response', 'source_document']
Ordered Q/A identical across the two files: True

Volume corpus (/home/group157/domain_corpus): identical 30 | differs 0 | absent on volume 0 | extra on volume 0
Volume Q/A file identical to repository copy: True

Split exports: train 957 (expected 957) | eval 239 (expected 239) | overlap 0 | union equals full file: True

>> [OK] Data integrity checked.


In [23]:
# Stage 0D: Model artifacts, checked against the committed models/MANIFEST.sha256
print("Stage 0D: model artifacts")
print("-" * 40)
model_expected = {}
for line in MODELS_MANIFEST.read_text(encoding="utf-8").splitlines():
    if line.strip():
        digest, rel = line.split("  ", 1)
        model_expected[rel] = digest

def check_dir(label, directory, subdir):
    # Compare every manifest entry under `subdir/` with the files in `directory`.
    if not directory.exists():
        print(f"{label:<28}MISSING: {directory}")
        return "FAIL", "directory missing"
    bad, absent, n = [], [], 0
    entries = [(rel, d) for rel, d in model_expected.items() if rel.startswith(subdir + "/")]
    print(f"{label}: {directory}")
    for k, (rel, digest) in enumerate(entries, start=1):
        n += 1
        p = directory / rel.split("/", 1)[1]
        size = human(p.stat().st_size) if p.exists() else "-"
        print(f"  [{k}/{len(entries)}] checking {rel.split('/', 1)[1]} ({size}) ...", end=" ", flush=True)
        if not p.exists():
            absent.append(rel)
            print("MISSING", flush=True)
        elif sha256_file(p) != digest:
            bad.append(rel)
            print("CHANGED", flush=True)
        else:
            print("OK", flush=True)
    print(f"{'':<2}files {n} | missing {len(absent)} | changed {len(bad)}")
    for r in absent + bad:
        print("   !", r)
    return ("PASS" if not (bad or absent) else "FAIL"), f"{n} files, changed={len(bad)}, missing={len(absent)}"

# Repository adapter (authoritative copy used for generation in Stage 6)
STAGE0["adapter_repo"] = check_dir("Adapter (repository)", ADAPTER_DIR, "qlora_finance_adapter")
ac = json.loads((ADAPTER_DIR / "adapter_config.json").read_text())
print("   adapter_config:", {k: ac.get(k) for k in ["base_model_name_or_path", "r", "lora_alpha", "target_modules", "lora_dropout"]})

# CPT model (weights are not in git; this hashes the 1.45 GB weight file, which takes a few seconds)
STAGE0["cpt_model"] = check_dir("CPT model", CPT_DIR, "cpt_model")
cfg_p = CPT_DIR / "config.json"
if cfg_p.exists():
    cfg = json.loads(cfg_p.read_text())
    print("   cpt config:", {k: cfg.get(k) for k in ["architectures", "vocab_size", "max_position_embeddings",
                                                  "hidden_size", "num_hidden_layers", "pad_token_id", "eos_token_id"]})

# Volume adapter copy (cross-check only; optional)
if PV_ADAPTER_DIR.exists():
    STAGE0["adapter_volume"] = check_dir("Adapter (1A volume copy)", PV_ADAPTER_DIR, "qlora_finance_adapter")
else:
    print(f"Adapter (1A volume copy)    not present at {PV_ADAPTER_DIR} (fine: the repository copy is used)")

aw = ADAPTER_DIR / "adapter_model.safetensors"
print(f"\nAdapter weights: {human(aw.stat().st_size)} | tokenizer saved with adapter: {(ADAPTER_DIR / 'tokenizer_config.json').exists()}")
print("\n>> [OK] Model artifacts verified against the committed manifest.")

Stage 0D: model artifacts
----------------------------------------
Adapter (repository): /home/group157/llm-assignment-2b/llm-rag/models/qlora_finance_adapter
OK[1/8] checking README.md (5.0 KB) ... 
  [2/8] checking adapter_config.json (635.0 B) ... OK
  [3/8] checking adapter_model.safetensors (6.3 MB) ... OK
  [4/8] checking merges.txt (455.5 KB) ... OK
  [5/8] checking special_tokens_map.json (864.0 B) ... OK
  [6/8] checking tokenizer.json (3.4 MB) ... OK
  [7/8] checking tokenizer_config.json (3.9 KB) ... OK
  [8/8] checking vocab.json (781.9 KB) ... OK
  files 8 | missing 0 | changed 0
   adapter_config: {'base_model_name_or_path': 'cpt_model', 'r': 16, 'lora_alpha': 32, 'target_modules': ['v_proj', 'q_proj'], 'lora_dropout': 0.05}
CPT model: /home/group157/model-backups/cpt_model_1a
OK[1/8] checking config.json (777.0 B) ... 
  [2/8] checking generation_config.json (111.0 B) ... OK
  [3/8] checking merges.txt (455.5 KB) ... OK
  [4/8] checking model.safetensors (1.3 GB) ... OK


In [ ]:
# Stage 0E: Load the corpus and save an input manifest
stage_header("Stage 0E: loading the corpus")
docs = {}
manifest = []
t0 = time.perf_counter()
print(f"Loading {len(corpus_files)} documents from {CORPUS_DIR} ...\n")
for i, p in enumerate(corpus_files, start=1):
    print(f"[{i:>2}/{len(corpus_files)}] loading {p.name[:80]} ...", end=" ", flush=True)
    text = p.read_text(encoding="utf-8")
    docs[p.name] = text
    manifest.append({"doc_id": p.name, "chars": len(text), "bytes": p.stat().st_size, "sha256": sha256_file(p)})
    print(f"OK ({len(text):,} chars)", flush=True)
print(f"\nAll {len(docs)} documents loaded in {time.perf_counter() - t0:.1f} s\n")

assert len(docs) == EXPECTED_DOCS, f"expected {EXPECTED_DOCS} documents, found {len(docs)}"

total_chars = sum(m["chars"] for m in manifest)
print("Stage 0E: corpus loaded")
print("-" * 40)
print(f"Documents      : {len(docs)}")
print(f"Total characters: {total_chars:,}")
print("\nFive largest / five smallest documents (characters):")
by_size = sorted(manifest, key=lambda m: m["chars"])
for m in by_size[:5] + by_size[-5:]:
    print(f"  {m['chars']:>10,}  {m['doc_id'][:90]}")

provenance = {
    "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "artifact_root": str(ARTIFACT_ROOT), "seed": SEED, "documents": manifest,
    "qa_file_sha256": sha256_file(QA_FILE), "curated_file_sha256": sha256_file(CURATED_FILE),
}
(WORK_DIR / "input_manifest.json").write_text(json.dumps(provenance, indent=2))
STAGE0["docs_loaded"] = ("PASS", f"{len(docs)} docs, {total_chars:,} chars")
print("\n>> [OK] Saved assignment_2b_outputs/input_manifest.json")

In [25]:
# Stage 0F: Model access and disk budget (no weights are downloaded here)
PLANNED_MODELS = [
    # (role, model id, stage, intended precision)
    ("embedder",            "BAAI/bge-small-en-v1.5",                  "Stage 2",  "fp32/fp16 on GPU"),
    ("cross-encoder",       "cross-encoder/ms-marco-MiniLM-L-6-v2",    "Stage 5",  "fp32/fp16 on GPU"),
    ("cross-encoder (alt)", "cross-encoder/ms-marco-MiniLM-L6-v2",     "Stage 5",  "fp32/fp16 on GPU"),
    ("generator fallback",  "Qwen/Qwen2.5-7B-Instruct",                "Stage 6",  "bf16 or 4-bit"),
    ("judge candidate",     "meta-llama/Llama-3.1-8B-Instruct",        "Stage 7",  "bf16 or 4-bit"),
]
print("Stage 0F: model access and download size (metadata only)")
print("-" * 60)
rows = []
try:
    from huggingface_hub import HfApi
    api = HfApi()
    for role, mid, stage, prec in PLANNED_MODELS:
        try:
            info = api.model_info(mid, files_metadata=True)
            weights = [s for s in info.siblings if s.rfilename.endswith((".safetensors", ".bin")) and s.size]
            size_gb = sum(s.size for s in weights) / 1e9
            gated = bool(getattr(info, "gated", False))
            rows.append((role, mid, stage, prec, "reachable", f"{size_gb:.2f} GB", "gated" if gated else "open"))
        except Exception as e:
            rows.append((role, mid, stage, prec, "NOT reachable", "?", type(e).__name__))
except Exception as e:
    print("huggingface_hub unavailable or network blocked:", repr(e))

for r in rows:
    print(f"{r[0]:<20}{r[1]:<40}{r[2]:<8}{r[3]:<18}{r[4]:<15}{r[5]:<10}{r[6]}")

free_gb = shutil.disk_usage(hf_home if hf_home.exists() else Path.home()).free / 1e9
print(f"\nFree disk where the HF cache lives: {free_gb:.1f} GB")
print("Reminder: a 4-bit runtime load still downloads the full-precision checkpoint unless a pre-quantized repo is used.")
print("Gated models (e.g. Llama) need an accepted licence and an HF token. Resolve this before Stage 7.")

ok = bool(rows) and any(r[4] == "reachable" for r in rows)
STAGE0["model_access"] = ("PASS" if ok else "WARN", f"{sum(r[4]=='reachable' for r in rows)}/{len(rows)} reachable")
print("\n>> [OK] Access check complete. Nothing downloaded.")

Stage 0F: model access and download size (metadata only)
------------------------------------------------------------
embedder            BAAI/bge-small-en-v1.5                  Stage 2 fp32/fp16 on GPU  reachable      0.27 GB   open
cross-encoder       cross-encoder/ms-marco-MiniLM-L-6-v2    Stage 5 fp32/fp16 on GPU  reachable      0.30 GB   open
cross-encoder (alt) cross-encoder/ms-marco-MiniLM-L6-v2     Stage 5 fp32/fp16 on GPU  reachable      0.30 GB   open
generator fallback  Qwen/Qwen2.5-7B-Instruct                Stage 6 bf16 or 4-bit     reachable      15.23 GB  open
judge candidate     meta-llama/Llama-3.1-8B-Instruct        Stage 7 bf16 or 4-bit     reachable      16.06 GB  gated

Free disk where the HF cache lives: 2459.8 GB
Reminder: a 4-bit runtime load still downloads the full-precision checkpoint unless a pre-quantized repo is used.
Gated models (e.g. Llama) need an accepted licence and an HF token. Resolve this before Stage 7.

>> [OK] Access check complete. Nothing dow

In [26]:
# Stage 0G: Gate 0 summary
print("Stage 0 gate summary")
print("-" * 60)
for k, (status, detail) in STAGE0.items():
    print(f"{status:<8}{k:<30}{detail}")

hard_fail = [k for k, (s, _) in STAGE0.items() if s == "FAIL"]
print()
if hard_fail:
    print("Gate 0 NOT passed. Resolve:", hard_fail)
else:
    print("No hard failures. Review any WARN / PENDING rows with the reviewer before starting Stage 1.")

_ = (WORK_DIR / "stage0_summary.json").write_text(json.dumps(STAGE0, indent=2))   # "_ =" stops Jupyter echoing the byte count
print(f"Saved {WORK_DIR.name}/stage0_summary.json")

Stage 0 gate summary
------------------------------------------------------------
PASS    resources                     gpu=NVIDIA L40S
PASS    repo_data_manifest            34 files checked
PASS    corpus                        30 files
PASS    qa_rows                       1196 rows
PASS    volume_corpus_matches_repo    same=30 diff=0
PASS    volume_qa_matches_repo        
PASS    split_exports                 957/239
PASS    adapter_repo                  8 files, changed=0, missing=0
PASS    cpt_model                     8 files, changed=0, missing=0
PASS    adapter_volume                8 files, changed=0, missing=0
PASS    docs_loaded                   30 docs, 7,160,100 chars
PASS    model_access                  5/5 reachable

No hard failures. Review any WARN / PENDING rows with the reviewer before starting Stage 1.
Saved assignment_2b_outputs/stage0_summary.json


---
## Stage 1: Query set and answer spans

- **Queries:** ten questions with expected answers, taken unchanged from the 239-row evaluation split of 1A (none from the 957 SFT training pairs). Each keeps its row id and a SHA-256 of its Q/A pair.
- **Answer spans:** each query has a raw passage from its source document that supports the full expected answer, recorded with the source filename and character offsets. Spans are frozen before retrieval and are not changed afterwards.
- **Hit rule (Parts A and B):** after lower-casing and collapsing whitespace, the span must be a substring of the retrieved chunk.
- **Limitations:** the evaluation split prevents reuse of an exact training pair but does not make the facts unseen, because CPT and other SFT pairs use the same corpus. The questions share vocabulary with their sources, which favours BM25. Q08 has near-miss net-worth passages that do not count as hits.
- **Unanswerable query:** U01 is an out-of-domain question kept separate from the ten and excluded from their metrics. It is used in D1.3 to test the fallback "Not found in the documents".

In [28]:
# Stage 1A: Evaluation membership and provenance
import re, math
# Rule: the graded queries must come from the held-out evaluation split, never from the SFT training pairs.
STAGE1 = {}
curated_rows = jsonl_rows(CURATED_FILE)     # 1,196 rows with source_document and evidence, in 1A row order
qa_rows      = jsonl_rows(QA_FILE)          # the file submitted in 1A (Q/A only)
train_rows   = jsonl_rows(TRAIN_FILE)
eval_rows    = jsonl_rows(EVAL_FILE)

qkey   = lambda r: (r["instruction"], r["response"])
qa_sha = lambda r: hashlib.sha256((r["instruction"] + "\n" + r["response"]).encode("utf-8")).hexdigest()

assert [qkey(r) for r in curated_rows] == [qkey(r) for r in qa_rows], "curated file must keep the 1A row order"
row_id_of = {}
for i, r in enumerate(curated_rows):
    assert qkey(r) not in row_id_of, f"duplicate Q/A pair at row {i}"
    row_id_of[qkey(r)] = i                  # row id = position in instruction_dataset.jsonl

train_keys, eval_keys = set(map(qkey, train_rows)), set(map(qkey, eval_rows))
assert len(train_rows) == EXPECTED_QA["train"] == len(train_keys), "train split size"
assert len(eval_rows)  == EXPECTED_QA["eval"]  == len(eval_keys),  "eval split size"
assert not (train_keys & eval_keys), "train and eval share a Q/A pair"
assert (train_keys | eval_keys) == set(row_id_of), "splits must cover the full 1,196 rows"

# Independent re-derivation of the 1A split (Stage 8B/8C of 1A): same seed, same shuffle, 80 / 20
records = list(qa_rows)
random.Random(SEED).shuffle(records)
cut = round(len(records) * 0.80)
assert [qkey(r) for r in records[:cut]] == [qkey(r) for r in train_rows], "train export differs from the 1A split"
assert [qkey(r) for r in records[cut:]] == [qkey(r) for r in eval_rows],  "eval export differs from the 1A split"

eval_docs = {}
for r in eval_rows:
    d = curated_rows[row_id_of[qkey(r)]]["source_document"]
    eval_docs[d] = eval_docs.get(d, 0) + 1
print("Stage 1A: evaluation pool")
print("-" * 60)
print(f"curated rows {len(curated_rows)} | train {len(train_rows)} | eval {len(eval_rows)} | train/eval overlap 0")
print(f"split re-derived from random.Random({SEED}).shuffle, cut at {cut}: exports match")
print(f"evaluation questions span {len(eval_docs)} of {EXPECTED_DOCS} documents; largest sources:")
for d, n in sorted(eval_docs.items(), key=lambda kv: -kv[1])[:5]:
    print(f"  {n:>3}  {d[:85]}")
STAGE1["eval_membership"] = ("PASS", f"{len(eval_rows)} eval rows, split re-derived")

Stage 1A: evaluation pool
------------------------------------------------------------
curated rows 1196 | train 957 | eval 239 | train/eval overlap 0
split re-derived from random.Random(42).shuffle, cut at 957: exports match
evaluation questions span 30 of 30 documents; largest sources:
   29  SEBI - Compliance with the provisions of the Securities and Exchange Board of India (
   23  RBI - Annual Report 2025-26.txt
   18  RBI - Annual Report 2023-24.txt
   17  SEBI Infrastructure Investment Trusts.txt
   14  SEBI Issue and Listing of Municipal Debt Securities.txt


In [ ]:
# Stage 1B: Ten evaluation questions with sufficient raw answer spans
stage_header("Stage 1B: ten evaluation queries and answer spans")
import textwrap
import pandas as pd

def normalise_with_map(text):
    """Lower-case and collapse whitespace runs to one space.
    Returns (normalised text, map) where map[j] is the raw index of normalised character j."""
    out, idx, in_space = [], [], False
    for i, ch in enumerate(text):
        if ch.isspace():
            if not in_space:
                out.append(" "); idx.append(i)
            in_space = True
        else:
            low = ch.lower()
            out.extend(low); idx.extend([i] * len(low))
            in_space = False
    return "".join(out), idx

# row_id = position in instruction_dataset.jsonl. `span` is copied verbatim from the source document
# (line breaks and all). `answer_keys` are facts of the expected answer that the span must contain.
SELECTION = [
    dict(qid='Q01', row_id=243, type='numeric limit',
         answer_keys=['₹25 lakh', 'maximum value per transaction'],
         span='In respect of inward or outward transactions \nprocessed by a PA - CB, the maximum value per transaction shall be ₹25 lakh.'),
    dict(qid='Q02', row_id=1069, type='where / account type',
         answer_keys=['escrow', 'scheduled commercial bank'],
         span='Non-bank PPI issuer is required to maintain the outstanding balance in an escrow \naccount with any scheduled commercial bank. An additional escrow account may be \nmaintained with a different scheduled commercial bank at the discretion of the PPI issuer.'),
    dict(qid='Q03', row_id=1093, type='statistic',
         answer_keys=['34%', '83%'],
         span='UPI has been the most significant contributor to the growth of digital payments \nin India. Its contribution to digital payments volume surged from 34% in CY-2019 to an \nimpressive 83% in CY-2024'),
    dict(qid='Q04', row_id=1113, type='prohibition',
         answer_keys=['no bank or system provider', 'any charge'],
         span='Notwithstanding anything contained in this Act, no bank or system provider shall impose, whether \ndirectly or indirectly, any charge upon a person making or receiving a payment by using the electronic \nmodes of payment prescribed under section 269-SU of the Income Tax Act, 1961 (43 of 1961).'),
    dict(qid='Q05', row_id=1149, type='definition and duty',
         answer_keys=['first loss default guarantee', 'securitisation of standard assets'],
         span='As regards the industry practice of offering financial products involving contractual \nagreements such as First Loss Default Guarantee (FLDG) in which a third party \nguarantees to compensate up to a certain percentage of default in a loan portfolio\n\nof the RE, it is advised that REs shall adhere to the provisions of the Master \nDirection – Reserve Bank of India (Securitisation of Standard Assets) Directions, \n2021 dated September 24, 2021, especially, synthetic securitisation8 contained in \nPara (6)(c).'),
    dict(qid='Q06', row_id=1164, type='proposal',
         answer_keys=['small payment system providers', 'not require to be authorised'],
         span='Thus, the case for recognising \na new class of Small Payment System Providers (SPSPs) that would not require to be \nauthorised by RBI before carrying out their activities shall be explored.'),
    dict(qid='Q07', row_id=897, type='price rule',
         answer_keys=['offer price shall stand revised', 'highest price paid or payable'],
         span='(8) Where the acquirer has acquired or agreed to acquire whether by himself or through \nor with persons acting in concert with him any shares or voting rights in the target \ncompany during the offer period, whether by subscription or purchase, at a price\n\nhigher than the offer price, the offer price shall stand revised to the highest price \npaid or payable for any such acquisition:'),
    dict(qid='Q08', row_id=864, type='threshold (distractors)',
         answer_keys=['networth', 'fifty crore'],
         span='the asset management company has a networth of not less than rupees fifty crore \ndeployed in assets as may be specified by the Board:'),
    dict(qid='Q09', row_id=826, type='margin rule',
         answer_keys=['elm of 1%', 'liquid assets of the clearing member'],
         span='ELM of 1% on gross open positions \nshall be levied and shall be deducted from the liquid assets of the \nclearing member on an online, real time basis.'),
    dict(qid='Q10', row_id=118, type='settlement mode',
         answer_keys=['only by way of cash'],
         span='the company shall pay the consideration only by way of cash;'),
]

norm_cache, frozen_queries = {}, []
for s in SELECTION:
    r = curated_rows[s["row_id"]]
    assert qkey(r) in eval_keys and qkey(r) not in train_keys, f"{s['qid']} is not an evaluation-only pair"
    name = r["source_document"]
    doc = docs[name]
    span = s["span"]
    assert span == span.strip(), f"{s['qid']}: span has leading/trailing whitespace"
    assert doc.count(span) == 1, f"{s['qid']}: raw span must occur exactly once in {name}"
    start = doc.find(span); end = start + len(span)
    assert doc[start:end] == span
    # Same span found through the normalised text and mapped back to raw offsets
    if name not in norm_cache:
        norm_cache[name] = normalise_with_map(doc)
    doc_norm, dmap = norm_cache[name]
    span_norm, _ = normalise_with_map(span)
    assert span_norm in doc_norm, f"{s['qid']}: normalised span not found in normalised document"
    assert doc_norm.count(span_norm) == 1
    p = doc_norm.find(span_norm)
    assert (dmap[p], dmap[p + len(span_norm) - 1] + 1) == (start, end), f"{s['qid']}: offset mapping mismatch"
    missing = [k for k in s["answer_keys"] if k not in span_norm]
    assert not missing, f"{s['qid']}: span lacks answer facts {missing}"
    frozen_queries.append({
        "qid": s["qid"], "type": s["type"], "row_id": s["row_id"],
        "eval_position": [qkey(x) for x in eval_rows].index(qkey(r)),
        "query": r["instruction"], "expected_answer": r["response"], "qa_sha256": qa_sha(r),
        "source_document": name, "doc_sha256": hashlib.sha256(doc.encode("utf-8")).hexdigest(),
        "curated_evidence": r["evidence"],
        "answer_span": span, "span_norm": span_norm, "span_start": start, "span_end": end,
        "answer_keys": s["answer_keys"],
    })

assert len(frozen_queries) == 10 and len({q["row_id"] for q in frozen_queries}) == 10
assert len({q["qid"] for q in frozen_queries}) == 10
assert all(q["span_norm"] in norm_cache[q["source_document"]][0] for q in frozen_queries)
n_docs = len({q["source_document"] for q in frozen_queries})
print(f"Checked {len(frozen_queries)} queries: all are evaluation-only pairs; every normalised span is present in its source")
print(f"document and maps back to its raw offsets; every span contains the answer facts. Source documents used: {n_docs}\n")

table = pd.DataFrame([{
    "qid": q["qid"], "row_id": q["row_id"], "type": q["type"],
    "source_document": q["source_document"].replace(".txt", "")[:46],
    "span (start-end)": f"{q['span_start']:,}-{q['span_end']:,}", "span chars": len(q["answer_span"]),
} for q in frozen_queries])
display(table) if "display" in globals() else print(table.to_string(index=False))

for q in frozen_queries:
    print("=" * 100)
    print(f"{q['qid']}  (row {q['row_id']}, {q['type']})  source: {q['source_document']}")
    print(textwrap.fill("QUERY   : " + q["query"], 100, subsequent_indent="          "))
    print(textwrap.fill("EXPECTED: " + q["expected_answer"], 100, subsequent_indent="          "))
    print(textwrap.fill("SPAN    : " + " ".join(q["answer_span"].split()), 100, subsequent_indent="          "))
    print(f"OFFSETS : characters {q['span_start']:,} to {q['span_end']:,} of {len(docs[q['source_document']]):,}")
STAGE1["ten_queries"] = ("PASS", f"{len(frozen_queries)} queries, {n_docs} documents")

In [ ]:
# Stage 1C: One clearly unanswerable query (kept separate from the ten)
stage_header("Stage 1C: unanswerable query U01")
UNANSWERABLE = {
    "qid": "U01",
    "query": "Who won the FIFA World Cup in 2018?",
    "category": "out-of-domain factual question",
    "expected_behaviour": "Not found in the documents",
    "why_absent": ("The corpus is 30 RBI/SEBI regulatory and statistical documents on payments, banking and securities markets. "
                   "None covers football or sports results, so the answer cannot be found or derived from it."),
}

def count_term(term):
    pat = re.compile(r"\b" + re.escape(term) + r"\b", re.IGNORECASE)
    per_doc = {n: len(pat.findall(t)) for n, t in docs.items()}
    return sum(per_doc.values()), sum(1 for c in per_doc.values() if c)

print("Check 1: exact keyword scan over all 30 documents (total hits, documents with a hit)")
checks = {}
for term in ["fifa", "football", "soccer", "croatia", "world cup", "champion", "winner", "france"]:
    checks[term] = count_term(term)
    print(f"  {term:<10} {checks[term][0]:>4} hits in {checks[term][1]:>2} documents")

print("\nCheck 2: the only 'world cup' hit, read in context (a near-miss, not an answer)")
for n, t in docs.items():
    for m in re.finditer(r"world\s+cup", t, flags=re.IGNORECASE):
        print(f"  [{n}]")
        print("  ..." + " ".join(t[max(0, m.start() - 160): m.end() + 60].split()) + "...")

# Check 3: best lexical matches among all paragraphs (plain word-overlap with inverse paragraph frequency;
# an inspection aid only, not the retrieval method used later)
q_terms = ["fifa", "world", "cup", "2018"]
paras = [(n, p) for n, t in docs.items() for p in re.split(r"\n\s*\n", t) if len(p.split()) >= 8]
tok = lambda s: set(re.findall(r"[a-z0-9]+", s.lower()))
para_tokens = [tok(p) for _, p in paras]
df = {w: sum(1 for ts in para_tokens if w in ts) for w in q_terms}
idf = {w: math.log((len(paras) + 1) / (df[w] + 1)) + 1 for w in q_terms}
scored = sorted(((sum(idf[w] for w in q_terms if w in ts), i) for i, ts in enumerate(para_tokens)), reverse=True)
print(f"\nCheck 3: paragraph counts for query words {df} of {len(paras):,} paragraphs. Best 3 lexical matches:")
for score, i in scored[:3]:
    n, p = paras[i]
    print(f"  score {score:.2f} | {n[:48]} | " + " ".join(p.split())[:170] + "...")

# Assertions: the answer cannot be in the corpus
assert checks["fifa"][0] == 0 and checks["football"][0] == 0 and checks["soccer"][0] == 0
assert not any(re.search(r"world\s+cup", p, re.I) and re.search(r"\b(won|winner|champion|champions|final)\b", p, re.I)
               for _, p in paras), "a paragraph pairs 'world cup' with a winner term"
print("\nAbsence argument: no FIFA/football/soccer text exists; the one 'World Cup' mention is an RBI public-awareness campaign")
print("run during the ICC cricket event, with no winner or year; the best lexical matches are unrelated regulatory text.")
print("Keywords alone do not prove absence, so D1.3 will also inspect the retrieved and reranked chunks for U01.")
UNANSWERABLE["absence_checks"] = {k: {"hits": v[0], "documents": v[1]} for k, v in checks.items()}
STAGE1["unanswerable"] = ("PASS", UNANSWERABLE["query"])

In [ ]:
# Stage 1D: Freeze the query set and save it for reuse in Parts A to D
stage_header("Stage 1D: frozen query set")
FROZEN_REPO_FILE = REPO_ROOT / "eval" / "query_set_frozen.json"        # committed: the reviewable, immutable copy
FROZEN_WORK_FILE = WORK_DIR / "query_set_frozen.json"                  # working copy on the volume

content = {"queries": frozen_queries, "unanswerable": UNANSWERABLE}
freeze_sha = hashlib.sha256(json.dumps(content, sort_keys=True, ensure_ascii=False).encode("utf-8")).hexdigest()
payload = {
    "description": "Frozen evaluation set for Assignment 2B (Group 157). Ten unchanged questions from the 239-row 1A evaluation split, "
                   "with raw answer spans, plus one unanswerable query kept out of all metrics over the ten.",
    "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "seed": SEED, "split": {"train": len(train_rows), "eval": len(eval_rows)},
    "hit_rule": "lower-case and collapse whitespace in both span and chunk; hit if span_norm is a substring of the chunk",
    "eval_file_sha256": sha256_file(EVAL_FILE), "curated_file_sha256": sha256_file(CURATED_FILE),
    "freeze_sha256": freeze_sha, **content,
}

if FROZEN_REPO_FILE.exists():                      # already frozen: the new build must be identical
    stored = json.loads(FROZEN_REPO_FILE.read_text(encoding="utf-8"))
    assert stored["freeze_sha256"] == freeze_sha, "frozen query set differs from the committed eval/query_set_frozen.json"
    print("Committed eval/query_set_frozen.json already exists and matches this build (freeze hash identical).")
else:
    FROZEN_REPO_FILE.parent.mkdir(parents=True, exist_ok=True)
    _ = FROZEN_REPO_FILE.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
    print("Wrote eval/query_set_frozen.json (first freeze). Commit this file.")
_ = FROZEN_WORK_FILE.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")

def load_query_set(path=FROZEN_REPO_FILE):
    """Later stages call this instead of re-selecting queries. Returns (ten queries, unanswerable query)."""
    d = json.loads(Path(path).read_text(encoding="utf-8"))
    check = hashlib.sha256(json.dumps({"queries": d["queries"], "unanswerable": d["unanswerable"]},
                                      sort_keys=True, ensure_ascii=False).encode("utf-8")).hexdigest()
    assert check == d["freeze_sha256"], "query set was modified after freezing"
    return d["queries"], d["unanswerable"]

QUERIES, UNANS = load_query_set()
assert [q["qid"] for q in QUERIES] == [q["qid"] for q in frozen_queries] and UNANS["qid"] == "U01"
assert all(q["span_norm"] in norm_cache[q["source_document"]][0] for q in QUERIES)
print(f"Reloaded {len(QUERIES)} queries + 1 unanswerable; freeze sha256 {freeze_sha[:16]}...")
print(f"Files: {FROZEN_REPO_FILE.relative_to(REPO_ROOT)}  and  {FROZEN_WORK_FILE}")
STAGE1["frozen"] = ("PASS", f"sha256 {freeze_sha[:12]}")

print("\nStage 1 gate summary")
print("-" * 60)
for k, (status, detail) in STAGE1.items():
    print(f"{status:<8}{k:<18}{detail}")
assert all(s == "PASS" for s, _ in STAGE1.values()), "Gate 1 not passed"
print("\nGate 1: all checks passed. Stop here and send the executed notebook to Codex before starting Stage 2.")

---
## Stage 2: Dense search (A1)

- **Bi-encoder:** `BAAI/bge-small-en-v1.5` embeds each chunk and each query independently into a 384-dimension vector.
- **Query prefix:** BGE instructs queries (not passages) to start with `Represent this sentence for searching relevant passages: `. Passages are embedded without a prefix.
- **Normalisation:** vectors are L2-normalised float32, so inner product equals cosine similarity.
- **Index:** FAISS `IndexFlatIP` is an exact (brute-force) inner-product index, not an approximate one.
- **Interface:** `DenseSearcher.build(chunks)` embeds and indexes any chunk list; `search(query, k)` returns ranked `chunk_id`, `score` and `text`. `build_dense_index` is the underlying function and returns the index, embeddings and build time.
- **Scope of this stage:** the interface is shown on a small labelled fixture of invented sentences. Fixture timings are not the A1 build time; Stage 3 builds the real 100- and 400-token chunk sets and reports the final-corpus time.

In [32]:
# Stage 2A: Load the dense embedder
import numpy as np
import faiss
import torch
from sentence_transformers import SentenceTransformer

EMBED_MODEL       = "BAAI/bge-small-en-v1.5"
BGE_QUERY_PREFIX  = "Represent this sentence for searching relevant passages: "   # queries only
EMBED_BATCH_SIZE  = 64
EMBED_DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"

t0 = time.perf_counter()
embedder = SentenceTransformer(EMBED_MODEL, device=EMBED_DEVICE)
load_seconds = time.perf_counter() - t0

EMBEDDER_FACTS = {
    "model": EMBED_MODEL,
    "revision": getattr(embedder[0].auto_model.config, "_commit_hash", None) or "unknown",
    "sentence_transformers": __import__("sentence_transformers").__version__,
    "vector_dim": embedder.get_sentence_embedding_dimension(),
    "max_seq_length_used": embedder.max_seq_length,
    "tokenizer_model_max_length": embedder.tokenizer.model_max_length,
    "device": EMBED_DEVICE,
    "batch_size": EMBED_BATCH_SIZE,
    "similarity": "cosine (inner product of L2-normalised float32 vectors)",
    "query_prefix": BGE_QUERY_PREFIX,
    "load_seconds": round(load_seconds, 2),
}
assert EMBEDDER_FACTS["vector_dim"] == 384
assert EMBEDDER_FACTS["max_seq_length_used"] <= 512

print("Stage 2A: embedder loaded")
print("-" * 60)
for k, v in EMBEDDER_FACTS.items():
    print(f"{k:<28}{v}")
_ = (WORK_DIR / "embedder_facts.json").write_text(json.dumps(EMBEDDER_FACTS, indent=2))
STAGE2 = {"embedder": ("PASS", f"{EMBED_MODEL} dim {EMBEDDER_FACTS['vector_dim']} on {EMBED_DEVICE}")}
print("\n>> [OK] Saved assignment_2b_outputs/embedder_facts.json")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/133M [00:00<?, ?B/s]

... (2 outputs left out)

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Stage 2A: embedder loaded
------------------------------------------------------------
model                       BAAI/bge-small-en-v1.5
revision                    5c38ec7c405ec4b44b94cc5a9bb96e735b38267a
sentence_transformers       3.3.1
vector_dim                  384
max_seq_length_used         512
tokenizer_model_max_length  512
device                      cuda
batch_size                  64
similarity                  cosine (inner product of L2-normalised float32 vectors)
query_prefix                Represent this sentence for searching relevant passages: 
load_seconds                16.74

>> [OK] Saved assignment_2b_outputs/embedder_facts.json


In [ ]:
# Stage 2B: Reusable dense search (any chunk list)
def embed_passages(texts, model, batch_size=EMBED_BATCH_SIZE):
    """Passages: no prefix. Returns contiguous float32 vectors, L2-normalised."""
    vecs = model.encode(list(texts), batch_size=batch_size, normalize_embeddings=True,
                        convert_to_numpy=True, show_progress_bar=False)
    return np.ascontiguousarray(vecs, dtype="float32")

def embed_query(query, model, prefix=BGE_QUERY_PREFIX):
    """Queries: BGE instruction prefix, then the same normalised float32 encoding."""
    vec = model.encode([prefix + query], batch_size=1, normalize_embeddings=True,
                       convert_to_numpy=True, show_progress_bar=False)
    return np.ascontiguousarray(vec, dtype="float32")

def build_dense_index(chunk_texts, model, batch_size=EMBED_BATCH_SIZE):
    """Embed all chunks and add them to an exact inner-product index.
    Returns (index, embeddings, build_seconds). build_seconds is wall-clock embedding + index.add."""
    t0 = time.perf_counter()
    emb = embed_passages(chunk_texts, model, batch_size)
    index = faiss.IndexFlatIP(emb.shape[1])
    index.add(emb)
    return index, emb, time.perf_counter() - t0

class DenseSearcher:
    """Dense retrieval over any chunk list. A chunk is a string or a dict with a "text" key and optional "chunk_id"."""
    def __init__(self, model, batch_size=EMBED_BATCH_SIZE, query_prefix=BGE_QUERY_PREFIX):
        self.model, self.batch_size, self.query_prefix = model, batch_size, query_prefix
        self.index = self.embeddings = self.chunks = None
        self.build_seconds = None

    def build(self, chunks):
        recs = []
        for i, c in enumerate(chunks):
            rec = dict(c) if isinstance(c, dict) else {"text": c}
            rec.setdefault("chunk_id", i)
            recs.append(rec)
        assert recs, "no chunks to index"
        self.index, self.embeddings, self.build_seconds = build_dense_index(
            [r["text"] for r in recs], self.model, self.batch_size)
        self.chunks = recs
        norms = np.linalg.norm(self.embeddings, axis=1)
        assert self.embeddings.dtype == np.float32 and np.allclose(norms, 1.0, atol=1e-3), "embeddings not normalised"
        assert self.index.ntotal == len(recs)
        return self

    def search(self, query, k=TOP_K_COMPARE):
        """Top-k chunks by cosine similarity: list of {rank, chunk_id, score, text, ...}. k is capped at the index size."""
        if self.index is None:
            raise RuntimeError("call build(chunks) first")
        if not isinstance(k, (int, np.integer)) or isinstance(k, bool) or k < 1:
            raise ValueError(f"k must be a positive integer, got {k!r}")
        k = min(int(k), self.index.ntotal)
        scores, ids = self.index.search(embed_query(query, self.model, self.query_prefix), k)
        return [{**self.chunks[i], "rank": r, "score": float(s)}
                for r, (s, i) in enumerate(zip(scores[0], ids[0]), start=1) if i >= 0]

stage_header("Stage 2B: dense search interface")
print("DenseSearcher defined (build / search; passages unprefixed, queries prefixed, vectors normalised float32)")
STAGE2["interface"] = ("PASS", "DenseSearcher + build_dense_index defined")

In [ ]:
# Stage 2C: Labelled fixture (mechanics check, not graded retrieval evidence)
stage_header("Stage 2C: labelled fixture")
FIXTURE_CHUNKS = [
    {"chunk_id": "F0", "text": "UPI is an instant payment system that lets customers transfer money between bank accounts using a mobile phone."},
    {"chunk_id": "F1", "text": "A company buying back its own shares must pay the consideration in cash and cannot withdraw the offer after announcing it."},
    {"chunk_id": "F2", "text": "Non-bank prepaid payment instrument issuers must keep customer balances in an escrow account with a scheduled commercial bank."},
    {"chunk_id": "F3", "text": "A loan is classified as non-performing when interest or principal remains overdue for more than ninety days."},
    {"chunk_id": "F4", "text": "Mutual fund schemes must publish their net asset value every business day so investors can see unit prices."},
    {"chunk_id": "F5", "text": "Exchanges levy extreme loss margin on commodity derivative positions to cover losses beyond normal price moves."},
    {"chunk_id": "F6", "text": "Payment aggregators must complete merchant onboarding checks, including know your customer verification, before settling funds."},
    {"chunk_id": "F7", "text": "The central bank publishes a financial stability report twice a year assessing risks to the banking system."},
]
FIXTURE_QUERIES = [   # (query, gold chunk_id): worded differently from the passage on purpose
    ("How can I send money instantly from my phone to another bank account?", "F0"),
    ("Is cash the only way a firm can pay shareholders in a share repurchase?", "F1"),
    ("Where must prepaid wallet issuers keep the money customers have loaded?", "F2"),
    ("When is a loan treated as having stopped performing?", "F3"),
    ("How often is the unit price of a mutual fund scheme disclosed?", "F4"),
    ("Which margin protects exchanges against extreme price moves in commodities?", "F5"),
]

fx = DenseSearcher(embedder, EMBED_BATCH_SIZE).build(FIXTURE_CHUNKS)
print(f"FIXTURE build time (not the graded A1 time): {fx.build_seconds*1000:.1f} ms for {len(FIXTURE_CHUNKS)} chunks "
      f"on {EMBED_DEVICE}, batch size {EMBED_BATCH_SIZE}\n")

# 1) Normalisation and index type
norms = np.linalg.norm(fx.embeddings, axis=1)
print(f"embeddings: shape {fx.embeddings.shape}, dtype {fx.embeddings.dtype}, norm min/max {norms.min():.6f}/{norms.max():.6f}")
assert fx.embeddings.shape == (8, EMBEDDER_FACTS["vector_dim"]) and fx.embeddings.dtype == np.float32
assert np.allclose(norms, 1.0, atol=1e-4)
assert isinstance(fx.index, faiss.IndexFlatIP) and fx.index.metric_type == faiss.METRIC_INNER_PRODUCT and fx.index.ntotal == 8
qv = embed_query(FIXTURE_QUERIES[0][0], embedder)
assert abs(float(np.linalg.norm(qv)) - 1.0) < 1e-4 and qv.dtype == np.float32

# 2) Prefix goes on queries only
raw_q = embedder.encode([FIXTURE_QUERIES[0][0]], normalize_embeddings=True, convert_to_numpy=True)
assert not np.allclose(raw_q, qv, atol=1e-4), "query prefix had no effect"
assert np.allclose(embed_passages([FIXTURE_CHUNKS[0]["text"]], embedder)[0], fx.embeddings[0], atol=1e-4), "passages must be unprefixed"

# 3) Ranking equals brute-force cosine; scores sorted; deterministic
for q, gold in FIXTURE_QUERIES:
    res = fx.search(q, k=8)
    brute = (embed_query(q, embedder) @ fx.embeddings.T)[0]
    by_id = {c["chunk_id"]: brute[i] for i, c in enumerate(FIXTURE_CHUNKS)}
    assert all(abs(r["score"] - by_id[r["chunk_id"]]) < 1e-5 for r in res), "returned score differs from brute-force cosine"
    assert np.allclose([r["score"] for r in res], np.sort(brute)[::-1], atol=1e-5)
    assert all(a["score"] >= b["score"] for a, b in zip(res, res[1:]))
    assert res == fx.search(q, k=8), "search is not deterministic"
print("ranking: matches brute-force cosine for all fixture queries; scores non-increasing; repeat search identical")

# 4) k bounds
assert [len(fx.search(FIXTURE_QUERIES[0][0], k=k)) for k in (1, 3, 8, 100)] == [1, 3, 8, 8]
for bad in (0, -1, 2.5, "3", True):
    try:
        fx.search("test", k=bad); raise AssertionError(f"k={bad!r} should be rejected")
    except ValueError:
        pass
print("top-k bounds: k=1,3,8 return 1,3,8 results; k=100 is capped at 8; k=0, -1, 2.5, '3', True raise ValueError")

# 5) Labelled retrieval (each gold chunk should be in the top 3)
print("\nFixture retrieval (top 3):")
top1 = 0
for q, gold in FIXTURE_QUERIES:
    res = fx.search(q, k=3)
    ids = [r["chunk_id"] for r in res]
    top1 += ids[0] == gold
    print(f"  gold {gold} | top-3 " + ", ".join(f"{r['chunk_id']} ({r['score']:.3f})" for r in res) + f" | {q}")
    assert gold in ids, f"gold {gold} not in top 3 for: {q}"
print(f"top-1 correct {top1}/{len(FIXTURE_QUERIES)}; gold in top-3 {len(FIXTURE_QUERIES)}/{len(FIXTURE_QUERIES)}")

print("\nExample result record for the first query (k=2):")
for r in fx.search(FIXTURE_QUERIES[0][0], k=2):
    print(f"  rank {r['rank']} | {r['chunk_id']} | score {r['score']:.4f} | {r['text'][:90]}...")
STAGE2["fixture_checks"] = ("PASS", f"top-1 {top1}/{len(FIXTURE_QUERIES)}, normalisation, ranking, k bounds")

In [35]:
# Stage 2D: Gate 2 summary
print("Stage 2 gate summary")
print("-" * 60)
for k, (status, detail) in STAGE2.items():
    print(f"{status:<8}{k:<18}{detail}")
assert all(s == "PASS" for s, _ in STAGE2.values()), "Gate 2 not passed"
print(f"\nEmbedder: {EMBEDDER_FACTS['model']} @ {EMBEDDER_FACTS['revision'][:12]} | dim {EMBEDDER_FACTS['vector_dim']} | "
      f"limit {EMBEDDER_FACTS['max_seq_length_used']} tokens | {EMBEDDER_FACTS['device']} | batch {EMBEDDER_FACTS['batch_size']}")
print("Fixture timing is a mechanics check only. The graded A1 build time is measured in Stage 3 on the final 100- and 400-token chunk sets.")
print("\nGate 2: all checks passed. Stop here and send the executed notebook to Codex before starting Stage 3.")

Stage 2 gate summary
------------------------------------------------------------
PASS    embedder          BAAI/bge-small-en-v1.5 dim 384 on cuda
PASS    interface         DenseSearcher + build_dense_index defined
PASS    fixture_checks    top-1 6/6, normalisation, ranking, k bounds

Embedder: BAAI/bge-small-en-v1.5 @ 5c38ec7c405e | dim 384 | limit 512 tokens | cuda | batch 64
Fixture timing is a mechanics check only. The graded A1 build time is measured in Stage 3 on the final 100- and 400-token chunk sets.

Gate 2: all checks passed. Stop here and send the executed notebook to Codex before starting Stage 3.


---
## Stage 3: Chunk size 100 vs 400 tokens (A2)

- **Chunking:** each document is tokenised with the embedder's tokenizer (`BAAI/bge-small-en-v1.5`, no special tokens) and cut into consecutive windows of N tokens with no overlap. Chunk text is the raw document substring between the first and last token offsets of the window, so every chunk keeps its document, raw character offsets and a unique ID (`c<N>-d<doc>-<chunk>`). The last chunk of a document may be shorter.
- **Length check:** each stored chunk is re-tokenised with special tokens and must fit the 512-token encoder limit. Cutting inside a word can change the re-tokenised count slightly; the differences are reported.
- **Evaluation:** the same ten frozen queries are searched on each chunk set (top 3). A hit means a top-3 chunk contains the frozen span under the Stage 1 hit rule. Spans that fall across a chunk boundary cannot be hit at that size; they are reported, not changed.
- **Metrics:** total chunks, hit@3, mean tokens in the top-3 chunks (embedder tokenizer, no special tokens), full-corpus build time (embedding plus `index.add`, wall clock; model load and fixture time excluded).
- **Tie policy:** results are ordered by score, then by chunk position, so repeated searches and later stages compare identically. Duplicate chunk IDs are rejected.
- **Decision rule (Parts B to D):** the larger hit@3 wins; on a tie, the size with the smaller mean generator prompt (SmolLM2 tokenizer). The generator context check uses the model's position limit; the 512 SFT length is a training setting, not a limit.
- **Limit of the evidence:** ten queries give 10% steps, so one query changes hit@3 by ten points.

In [ ]:
# Stage 3A: Chunk the corpus (100 and 400 embedder tokens, no overlap)
stage_header("Stage 3A: chunking the corpus")
from transformers import AutoTokenizer

STAGE3 = {}
STAGE3_DIR = WORK_DIR / "stage3"
STAGE3_DIR.mkdir(parents=True, exist_ok=True)

# Same tokenizer files as the loaded embedder (pinned revision), used only for chunking and token counts
chunk_tok = AutoTokenizer.from_pretrained(EMBED_MODEL, revision=EMBEDDER_FACTS["revision"])
assert chunk_tok.is_fast, "a fast tokenizer is needed for character offsets"
chunk_tok.model_max_length = 10**9          # whole documents are tokenised at once; the 512 limit is checked per stored chunk
ENC_LIMIT = EMBEDDER_FACTS["max_seq_length_used"]
doc_names = sorted(docs)                    # stable document order

def chunk_document(doc_index, name, text, n):
    offs = chunk_tok(text, add_special_tokens=False, return_offsets_mapping=True)["offset_mapping"]
    out = []
    for ci, i in enumerate(range(0, len(offs), n)):
        win = offs[i:i + n]
        s, e = win[0][0], win[-1][1]
        out.append({"chunk_id": f"c{n}-d{doc_index:02d}-{ci:05d}", "doc_id": name, "doc_index": doc_index,
                    "chunk_index": ci, "char_start": s, "char_end": e, "n_tokens": len(win), "text": text[s:e]})
    return out

CHUNKS, CHUNK_TIME = {n: [] for n in CHUNK_SIZES}, {n: 0.0 for n in CHUNK_SIZES}
print(f"Chunking {len(doc_names)} documents with tokenizer {EMBED_MODEL} @ {EMBEDDER_FACTS['revision'][:12]}\n")
for di, name in enumerate(doc_names):
    per = []
    for n in CHUNK_SIZES:
        t0 = time.perf_counter()
        cs = chunk_document(di, name, docs[name], n)
        CHUNK_TIME[n] += time.perf_counter() - t0
        CHUNKS[n] += cs
        per.append(f"{n}-token {len(cs):>6,}")
    print(f"[{di+1:>2}/{len(doc_names)}] {name[:60]:<60} " + " | ".join(per), flush=True)

CHUNK_STATS = {}
for n in CHUNK_SIZES:
    chunks = CHUNKS[n]
    # IDs, offsets, order
    ids = [c["chunk_id"] for c in chunks]
    assert len(ids) == len(set(ids)), f"duplicate chunk IDs at size {n}"
    by_doc = {}
    for c in chunks:
        assert docs[c["doc_id"]][c["char_start"]:c["char_end"]] == c["text"], "offsets do not reproduce the chunk text"
        by_doc.setdefault(c["doc_id"], []).append(c)
    uncovered = 0
    for name, cs in by_doc.items():
        assert all(a["char_end"] <= b["char_start"] for a, b in zip(cs, cs[1:])), "chunks overlap or are out of order"
        gaps = [docs[name][:cs[0]["char_start"]], docs[name][cs[-1]["char_end"]:]]
        gaps += [docs[name][a["char_end"]:b["char_start"]] for a, b in zip(cs, cs[1:])]
        uncovered += sum(1 for g in gaps for ch in g if not ch.isspace())
    # Re-tokenise every stored chunk exactly as the encoder will see it
    texts = [c["text"] for c in chunks]
    with_special = [len(x) for x in chunk_tok(texts, add_special_tokens=True)["input_ids"]]
    plain = [len(x) for x in chunk_tok(texts, add_special_tokens=False)["input_ids"]]
    for c, ws, pl in zip(chunks, with_special, plain):
        c["enc_tokens"], c["enc_tokens_special"] = pl, ws
    assert max(with_special) <= ENC_LIMIT, f"a {n}-token chunk exceeds the encoder limit"
    drift = [pl - c["n_tokens"] for c, pl in zip(chunks, plain)]
    short = sum(1 for c in chunks if c["n_tokens"] < n)
    norm_texts = {" ".join(t.lower().split()) for t in texts}
    CHUNK_STATS[n] = {
        "total_chunks": len(chunks), "documents": len(by_doc), "final_short_chunks": short,
        "nominal_tokens_min_mean_max": [min(c["n_tokens"] for c in chunks), round(float(np.mean([c["n_tokens"] for c in chunks])), 1), n],
        "encoder_len_with_special_min_mean_max": [min(with_special), round(float(np.mean(with_special)), 1), max(with_special)],
        "retokenised_differs_from_nominal": sum(1 for d in drift if d), "max_abs_drift": max(abs(d) for d in drift),
        "uncovered_non_whitespace_chars": uncovered, "duplicate_text_chunks": len(chunks) - len(norm_texts),
        "chunking_seconds": round(CHUNK_TIME[n], 2)}
    _ = (STAGE3_DIR / f"chunks_{n}.jsonl").write_text("\n".join(json.dumps(c, ensure_ascii=False) for c in chunks) + "\n", encoding="utf-8")

print("\nChunk checks (all assertions passed: unique IDs, offsets reproduce text, no overlap, encoder length <= 512)")
print("-" * 60)
for n in CHUNK_SIZES:
    print(f"\n{n}-token chunks")
    for k, v in CHUNK_STATS[n].items():
        print(f"  {k:<40}{v}")
print(f"\nSaved chunk files under {STAGE3_DIR.name}/ (chunks_100.jsonl, chunks_400.jsonl)")
STAGE3["chunks"] = ("PASS", " | ".join(f"{n}: {CHUNK_STATS[n]['total_chunks']:,}" for n in CHUNK_SIZES))

In [ ]:
# Stage 3B: Build a dense index for each chunk size (full corpus)
stage_header("Stage 3B: dense index build on the full corpus")

class StableDenseSearcher(DenseSearcher):
    """DenseSearcher with a fixed tie policy (score descending, then chunk position ascending) and a duplicate-ID guard."""
    TIE_MARGIN = 25
    def build(self, chunks):
        ids = [c["chunk_id"] for c in chunks]
        assert len(ids) == len(set(ids)), "duplicate chunk IDs"
        return super().build(chunks)

    def search(self, query, k=TOP_K_COMPARE):
        if self.index is None:
            raise RuntimeError("call build(chunks) first")
        if not isinstance(k, (int, np.integer)) or isinstance(k, bool) or k < 1:
            raise ValueError(f"k must be a positive integer, got {k!r}")
        n_total = self.index.ntotal
        k = min(int(k), n_total)
        q = embed_query(query, self.model, self.query_prefix)
        fetch = min(k + self.TIE_MARGIN, n_total)
        scores, ids = self.index.search(q, fetch)
        ranked = sorted(((float(s), int(i)) for s, i in zip(scores[0], ids[0]) if i >= 0), key=lambda t: (-t[0], t[1]))[:k]
        if fetch < n_total and float(scores[0][-1]) == ranked[-1][0]:      # a tie may continue beyond the fetched list: exact fallback
            full = (self.embeddings @ q[0]).astype("float32")
            order = np.lexsort((np.arange(n_total), -full))[:k]
            ranked = [(float(full[i]), int(i)) for i in order]
        return [{**self.chunks[i], "rank": r, "score": s, "index_id": i} for r, (s, i) in enumerate(ranked, start=1)]

SEARCHERS, BUILD = {}, {}
print(f"Embedding on {EMBED_DEVICE}, batch size {EMBED_BATCH_SIZE}. Model load ({EMBEDDER_FACTS['load_seconds']} s) and the 8-chunk fixture are NOT part of these times.\n")
for n in CHUNK_SIZES:
    print(f"Building the {n}-token index: embedding {len(CHUNKS[n]):,} chunks ...", end=" ", flush=True)
    s = StableDenseSearcher(embedder, EMBED_BATCH_SIZE).build(CHUNKS[n])
    SEARCHERS[n] = s
    BUILD[n] = {"chunk_size": n, "chunks": len(CHUNKS[n]), "build_seconds": round(s.build_seconds, 2),
                "chunks_per_second": round(len(CHUNKS[n]) / s.build_seconds, 1),
                "embedding_shape": list(s.embeddings.shape), "dtype": str(s.embeddings.dtype)}
    print(f"OK in {s.build_seconds:.1f} s", flush=True)
    assert s.index.ntotal == len(CHUNKS[n]) and isinstance(s.index, faiss.IndexFlatIP)
    np.save(STAGE3_DIR / f"embeddings_{n}.npy", s.embeddings)
    faiss.write_index(s.index, str(STAGE3_DIR / f"index_{n}.faiss"))

print("\nFull-corpus index build (embedding + index.add, wall clock; one warm run per size)")
print("-" * 60)
print(f"{'chunk size':>10} {'chunks':>9} {'build seconds':>14} {'chunks/s':>10}  embeddings")
for n in CHUNK_SIZES:
    b = BUILD[n]
    print(f"{n:>10} {b['chunks']:>9,} {b['build_seconds']:>14.2f} {b['chunks_per_second']:>10.1f}  {tuple(b['embedding_shape'])} {b['dtype']}")
print("\nSaved embeddings_<N>.npy and index_<N>.faiss under", STAGE3_DIR.name + "/")
STAGE3["indexes"] = ("PASS", " | ".join(f"{n}: {BUILD[n]['build_seconds']} s" for n in CHUNK_SIZES))

In [ ]:
# Stage 3C: Dense search on the ten frozen queries (top 3), both chunk sizes
stage_header("Stage 3C: dense retrieval on the ten frozen queries")
import pandas as pd

RESULTS, METRICS, norm_index = {}, {}, {}
for n in CHUNK_SIZES:
    chunks, searcher = CHUNKS[n], SEARCHERS[n]
    norm_index[n] = {i: normalise_with_map(c["text"])[0] for i, c in enumerate(chunks)}
    per_query = []
    for q in QUERIES:
        top = searcher.search(q["query"], k=20)
        top3 = top[:3]
        hit3 = any(q["span_norm"] in normalise_with_map(c["text"])[0] for c in top3)
        # Chunks that contain the span under the frozen hit rule (any document), and those that contain its raw offsets
        gold_ids = {chunks[i]["chunk_id"] for i, t in norm_index[n].items() if q["span_norm"] in t}
        src_gold = {c["chunk_id"] for c in chunks if c["doc_id"] == q["source_document"]
                    and c["char_start"] <= q["span_start"] and c["char_end"] >= q["span_end"]}
        assert src_gold <= gold_ids, f"{q['qid']}: offset containment is not covered by the text rule"
        overlap = [c for c in chunks if c["doc_id"] == q["source_document"]
                   and c["char_start"] < q["span_end"] and c["char_end"] > q["span_start"]]
        straddles = len(src_gold) == 0                  # the source passage is cut by a chunk boundary at this size
        elsewhere = len(gold_ids - src_gold)            # identical text found in other chunks (other positions or documents)
        gold_rank = next((c["rank"] for c in top if c["chunk_id"] in gold_ids), None)
        assert hit3 == (gold_rank is not None and gold_rank <= 3), f"{q['qid']}: text hit and containing-chunk rank disagree"
        rec = {"qid": q["qid"], "query": q["query"], "hit_at_3": hit3, "span_straddles_boundary": straddles,
               "gold_chunk_ids": sorted(gold_ids), "source_chunk_ids": sorted(src_gold), "other_chunks_with_same_text": elsewhere,
               "gold_rank_in_top20": gold_rank,
               "top3_enc_tokens": sum(c["enc_tokens"] for c in top3), "top3_chars": sum(len(c["text"]) for c in top3),
               "top3": [{k: c[k] for k in ("rank", "chunk_id", "score", "doc_id", "char_start", "char_end", "enc_tokens", "text")} for c in top3],
               "top20": [(c["chunk_id"], round(c["score"], 6)) for c in top]}
        if straddles:
            rec["boundary_detail"] = [(c["chunk_id"], c["char_start"], c["char_end"]) for c in overlap]
        per_query.append(rec)
    RESULTS[n] = per_query
    hits = sum(r["hit_at_3"] for r in per_query)
    METRICS[n] = {"chunk_size": n, "total_chunks": len(chunks), "hit_at_3": hits, "hit_at_3_pct": 100 * hits / len(QUERIES),
                  "gold_in_top20": sum(r["gold_rank_in_top20"] is not None for r in per_query),
                  "spans_straddling_boundary": sum(r["span_straddles_boundary"] for r in per_query),
                  "mean_top3_tokens": round(float(np.mean([r["top3_enc_tokens"] for r in per_query])), 1),
                  "mean_top3_chars": round(float(np.mean([r["top3_chars"] for r in per_query])), 1)}
    _ = (STAGE3_DIR / f"dense_results_{n}.json").write_text(json.dumps(per_query, indent=1, ensure_ascii=False), encoding="utf-8")

print("Per-query result (hit = a top-3 chunk contains the frozen span; rank = best rank of a chunk containing it, within the top 20)")
rows = []
for i, q in enumerate(QUERIES):
    row = {"qid": q["qid"], "source": q["source_document"].replace(".txt", "")[:34]}
    for n in CHUNK_SIZES:
        r = RESULTS[n][i]
        row[f"{n}: hit@3"] = "HIT" if r["hit_at_3"] else ("split" if not r["gold_chunk_ids"] else "miss")
        row[f"{n}: rank"] = r["gold_rank_in_top20"] if r["gold_rank_in_top20"] is not None else "-"
    rows.append(row)
print(pd.DataFrame(rows).to_string(index=False))
print("('split' = no chunk of this size contains the span; 'miss' = a chunk contains it but it is not in the top 3)")
for n in CHUNK_SIZES:
    extra = [(q["qid"], r["other_chunks_with_same_text"]) for q, r in zip(QUERIES, RESULTS[n]) if r["other_chunks_with_same_text"]]
    if extra:
        print(f"Note ({n}-token): identical span text also occurs in other chunks for {extra}; a hit on such a chunk counts under the hit rule.")

print("\nSummary")
print("-" * 60)
print(f"{'chunk size':>10} {'total chunks':>13} {'hit@3':>8} {'hit@3 %':>8} {'span in top20':>14} {'split spans':>12} {'mean top-3 tokens':>18}")
for n in CHUNK_SIZES:
    m = METRICS[n]
    print(f"{n:>10} {m['total_chunks']:>13,} {m['hit_at_3']:>5}/10 {m['hit_at_3_pct']:>7.0f}% {m['gold_in_top20']:>11}/10 {m['spans_straddling_boundary']:>9}/10 {m['mean_top3_tokens']:>18.1f}")
print("Token counts use the embedder tokenizer (BAAI/bge-small-en-v1.5) on each stored chunk, without special tokens.")

print("\nSpans split across chunk boundaries (spans are unchanged)")
any_split = False
for n in CHUNK_SIZES:
    for q, r in zip(QUERIES, RESULTS[n]):
        if r["span_straddles_boundary"]:
            any_split = True
            parts = "; ".join(f"{cid} [{s:,}-{e:,}]" for cid, s, e in r["boundary_detail"])
            print(f"  {n}-token | {q['qid']} span {q['span_start']:,}-{q['span_end']:,} overlaps {parts}")
if not any_split:
    print("  none")
print("\nSaved dense_results_100.json and dense_results_400.json")
STAGE3["retrieval"] = ("PASS", " | ".join(f"{n}: hit@3 {METRICS[n]['hit_at_3']}/10" for n in CHUNK_SIZES))

In [ ]:
# Stage 3D: Generator context budget, chunk-size decision and A1 facts
stage_header("Stage 3D: chunk-size decision and A1 facts")

gen_src = ADAPTER_DIR if (ADAPTER_DIR / "tokenizer.json").exists() else CPT_DIR
gen_tok = AutoTokenizer.from_pretrained(str(gen_src))
gen_cfg_path = CPT_DIR / "config.json"
GEN_MAX_POSITIONS = json.loads(gen_cfg_path.read_text()).get("max_position_embeddings") if gen_cfg_path.exists() else None
SFT_MAX_LENGTH = 512     # 1A SFT training setting, not a model limit

# Provisional grounded prompt (Stage 6 fixes the final wording); used only to size the context
PROMPT_HEAD = ("Answer the question using only the numbered context passages. Cite passages like [1]. "
               "If the answer is not in the passages, reply exactly: Not found in the documents.\n\n")
def provisional_prompt(question, chunk_texts):
    ctx = "\n\n".join(f"[{i}] {t}" for i, t in enumerate(chunk_texts, start=1))
    return f"{PROMPT_HEAD}Context:\n{ctx}\n\nQuestion: {question}\nAnswer:"

BUDGET = {}
for n in CHUNK_SIZES:
    lens = [len(gen_tok(provisional_prompt(r["query"], [c["text"] for c in r["top3"]]), add_special_tokens=False)["input_ids"])
            for r in RESULTS[n]]
    BUDGET[n] = {"mean_prompt_tokens": round(float(np.mean(lens)), 1), "max_prompt_tokens": max(lens),
                 "prompts_over_512": sum(l > SFT_MAX_LENGTH for l in lens),
                 "fits_model_positions": (max(lens) <= GEN_MAX_POSITIONS) if GEN_MAX_POSITIONS else None}

print(f"Generator tokenizer: {gen_src.name} (1A model). Model position limit: {GEN_MAX_POSITIONS}. 1A SFT max_length: {SFT_MAX_LENGTH} (a training setting).")
print("Prompt = provisional grounded template + top-3 chunks + question, counted in the generator tokenizer\n")
print(f"{'chunk size':>10} {'mean prompt tokens':>19} {'max prompt tokens':>18} {'prompts > 512':>14} {'fits model positions':>21}")
for n in CHUNK_SIZES:
    b = BUDGET[n]
    print(f"{n:>10} {b['mean_prompt_tokens']:>19.1f} {b['max_prompt_tokens']:>18} {b['prompts_over_512']:>11}/10 {str(b['fits_model_positions']):>21}")

# Decision rule: larger hit@3; on a tie, smaller mean generator prompt. A size whose prompts exceed the model's positions is excluded.
eligible = [n for n in CHUNK_SIZES if BUDGET[n]["fits_model_positions"] is not False]
assert eligible, "no chunk size fits the generator context"
CHOSEN_CHUNK_SIZE = sorted(eligible, key=lambda n: (-METRICS[n]["hit_at_3"], BUDGET[n]["mean_prompt_tokens"]))[0]
other = [n for n in CHUNK_SIZES if n != CHOSEN_CHUNK_SIZE][0]
c, o = METRICS[CHOSEN_CHUNK_SIZE], METRICS[other]
basis = ("a higher hit@3" if c["hit_at_3"] > o["hit_at_3"] else "equal hit@3 and a smaller mean generator prompt")
DECISION_TEXT = (f"Chunk size {CHOSEN_CHUNK_SIZE} tokens is used for Parts B to D: hit@3 {c['hit_at_3']}/10 versus {o['hit_at_3']}/10 for {other}, "
                 f"{c['spans_straddling_boundary']} versus {o['spans_straddling_boundary']} frozen spans split across chunk boundaries, "
                 f"mean top-3 embedder tokens {c['mean_top3_tokens']} versus {o['mean_top3_tokens']}, and mean generator prompt "
                 f"{BUDGET[CHOSEN_CHUNK_SIZE]['mean_prompt_tokens']} versus {BUDGET[other]['mean_prompt_tokens']} tokens. "
                 f"The rule selected it on {basis}. With ten queries, one query is 10 points.")
print("\nDecision")
print("-" * 60)
import textwrap
print(textwrap.fill(DECISION_TEXT, 100))
if BUDGET[CHOSEN_CHUNK_SIZE]["prompts_over_512"]:
    print(f"Note: {BUDGET[CHOSEN_CHUNK_SIZE]['prompts_over_512']} of 10 provisional prompts exceed the 512-token SFT length; "
          "Stage 6 tests the 1A generator on these prompts rather than assuming failure.")

b = BUILD[CHOSEN_CHUNK_SIZE]
A1_FACTS = {
    "Embedding model": EMBED_MODEL, "Model revision": EMBEDDER_FACTS["revision"],
    "Vector size": EMBEDDER_FACTS["vector_dim"],
    "Similarity": "cosine (inner product of L2-normalised float32 vectors, exact FAISS IndexFlatIP)",
    "Query prefix": "BGE instruction on queries only", "Batch size": EMBED_BATCH_SIZE, "Device": EMBED_DEVICE,
    "Chunk size (tokens, no overlap)": CHOSEN_CHUNK_SIZE, "Total chunks": b["chunks"],
    "Index build time (embedding + index.add, full corpus)": f"{b['build_seconds']} s",
    "Token counts": "embedder tokenizer, stored chunk, no special tokens",
    "Encoder limit": f"{ENC_LIMIT} tokens (max stored chunk {CHUNK_STATS[CHOSEN_CHUNK_SIZE]['encoder_len_with_special_min_mean_max'][2]} with special tokens)",
    "Tie policy": "score descending, then chunk position ascending"}
print("\nA1 facts table (chosen size)")
print("-" * 60)
for k, v in A1_FACTS.items():
    print(f"{k:<56}{v}")
STAGE3["decision"] = ("PASS", f"chosen {CHOSEN_CHUNK_SIZE} tokens")

In [ ]:
# Stage 3E: Save FINAL_CHUNKS and provenance, verify reuse, gate summary
stage_header("Stage 3E: freeze FINAL_CHUNKS and verify reuse")

FINAL_CHUNKS = CHUNKS[CHOSEN_CHUNK_SIZE]
assert len({c["chunk_id"] for c in FINAL_CHUNKS}) == len(FINAL_CHUNKS)
_ = (STAGE3_DIR / "final_chunks.jsonl").write_text("\n".join(json.dumps(c, ensure_ascii=False) for c in FINAL_CHUNKS) + "\n", encoding="utf-8")

summary = {
    "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "chosen_chunk_size": CHOSEN_CHUNK_SIZE, "decision": DECISION_TEXT, "decision_rule": "higher hit@3; tie -> smaller mean generator prompt tokens",
    "chunk_stats": CHUNK_STATS, "build": BUILD, "metrics": METRICS, "generator_budget": BUDGET,
    "generator_tokenizer": str(gen_src.name), "generator_max_positions": GEN_MAX_POSITIONS, "a1_facts": A1_FACTS,
    "provenance": {
        "embedder": EMBEDDER_FACTS, "chunk_tokenizer": EMBED_MODEL, "chunk_tokenizer_revision": EMBEDDER_FACTS["revision"],
        "chunking": {"sizes": CHUNK_SIZES, "overlap": 0, "unit": "embedder tokens, no special tokens", "document_order": "sorted file name",
                     "id_format": "c<N>-d<doc index>-<chunk index>"},
        "frozen_query_set_sha256": json.loads(FROZEN_REPO_FILE.read_text(encoding="utf-8"))["freeze_sha256"],
        "input_manifest_sha256": sha256_file(WORK_DIR / "input_manifest.json"),
        "corpus_documents": len(docs), "corpus_chars": sum(len(t) for t in docs.values()),
        "python": platform.python_version(), "torch": torch.__version__, "faiss": faiss.__version__},
}
_ = (STAGE3_DIR / "stage3_summary.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")
(STAGE3_DIR / "FINAL_CHUNKS_CONFIG.json").write_text(json.dumps(
    {"chunk_size": CHOSEN_CHUNK_SIZE, "chunks": len(FINAL_CHUNKS), "file": "final_chunks.jsonl",
     "embeddings": f"embeddings_{CHOSEN_CHUNK_SIZE}.npy", "index": f"index_{CHOSEN_CHUNK_SIZE}.faiss"}, indent=2))

saved = sorted(p for p in STAGE3_DIR.iterdir() if p.is_file() and p.name != "stage3_manifest.sha256")
_ = (STAGE3_DIR / "stage3_manifest.sha256").write_text("".join(f"{sha256_file(p)}  {p.name}\n" for p in saved))

def load_dense_searcher(n=None):
    """Rebuild a searcher from the saved chunks, embeddings and index without re-embedding (used by Stages 4 to 7)."""
    n = n or CHOSEN_CHUNK_SIZE
    chunks = [json.loads(l) for l in (STAGE3_DIR / f"chunks_{n}.jsonl").read_text(encoding="utf-8").splitlines() if l.strip()]
    s = StableDenseSearcher(embedder, EMBED_BATCH_SIZE)
    s.chunks, s.embeddings = chunks, np.load(STAGE3_DIR / f"embeddings_{n}.npy")
    s.index = faiss.read_index(str(STAGE3_DIR / f"index_{n}.faiss"))
    s.build_seconds = BUILD[n]["build_seconds"] if n in BUILD else None
    assert s.index.ntotal == len(chunks) == len(s.embeddings)
    assert len({c["chunk_id"] for c in chunks}) == len(chunks)
    return s

reloaded = load_dense_searcher(CHOSEN_CHUNK_SIZE)
for q, r in zip(QUERIES, RESULTS[CHOSEN_CHUNK_SIZE]):
    again = reloaded.search(q["query"], k=3)
    assert [c["chunk_id"] for c in again] == [c["chunk_id"] for c in r["top3"]], f"{q['qid']}: reloaded ranking differs"
    assert np.allclose([c["score"] for c in again], [c["score"] for c in r["top3"]], atol=1e-5)
print(f"Reload check: saved chunks + embeddings + index reproduce the in-memory top-3 for all {len(QUERIES)} queries ({CHOSEN_CHUNK_SIZE}-token set).")

print("\nSaved files (SHA-256 in stage3_manifest.sha256)")
print("-" * 60)
for p in saved:
    print(f"  {p.name:<28}{p.stat().st_size/1e6:>9.2f} MB  {sha256_file(p)[:16]}...")
print(f"\nFINAL_CHUNKS: {len(FINAL_CHUNKS):,} chunks of {CHOSEN_CHUNK_SIZE} tokens (final_chunks.jsonl). Use load_dense_searcher() in Stages 4 to 7.")
STAGE3["saved"] = ("PASS", f"{len(saved)} files, reload check passed")

print("\nStage 3 gate summary")
print("-" * 60)
for k, (status, detail) in STAGE3.items():
    print(f"{status:<8}{k:<14}{detail}")
assert all(s == "PASS" for s, _ in STAGE3.values()), "Gate 3 not passed"
print("\nGate 3: all checks passed. Stop here and send the executed notebook to Codex before starting Stage 4.")